# 03 - Inference Demo: Pipeline Two-Stage (A3-final + B2)

Vizualizare end-to-end pe imaginile din **test set** (`parks_detect_full`).
Imaginile sunt selectate **automat din dataset** - nu este nevoie sa adaugi imagini manual.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path('../..').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print('REPO_ROOT:', REPO_ROOT)

In [ ]:
import csv
from collections import Counter

import cv2
import matplotlib.pyplot as plt
import numpy as np

from ultralytics import YOLO
from src.detect_two_stage import detect_and_classify, draw_detections, classifier_names

print(f'Radacina proiectului: {REPO_ROOT}')

In [ ]:
# CONFIG

DETECTOR_PT   = REPO_ROOT / 'runs' / 'detect'   / 'parks-trash-A3-final' / 'weights' / 'best.pt'
CLASSIFIER_PT = REPO_ROOT / 'runs' / 'classify' / 'parks-cls-B2'        / 'weights' / 'best.pt'

SOURCE_DIR    = REPO_ROOT / 'datasets' / 'parks_detect_full' / 'images' / 'test'

DET_CONF      = 0.25
DET_IMGSZ     = 640
CLS_IMGSZ     = 224
MAX_LABELS    = 5
LINE_WIDTH    = 2
DEVICE        = '0'

OUTPUT_DIR    = REPO_ROOT / 'outputs' / 'two_stage_demo'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS    = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

print(f'Detector  : {DETECTOR_PT}')
print(f'Clasificor: {CLASSIFIER_PT}')
print(f'Sursa     : {SOURCE_DIR}')

---
## Incarcare modele

In [ ]:
assert DETECTOR_PT.exists(),   f'Detector lipsa: {DETECTOR_PT}'
assert CLASSIFIER_PT.exists(), f'Clasificator lipsa: {CLASSIFIER_PT}'

detector   = YOLO(str(DETECTOR_PT))
classifier = YOLO(str(CLASSIFIER_PT))
cls_names  = classifier_names(classifier)

print(f'Clase clasificator: {cls_names}')

---
## Inferenta pe imaginile test

In [ ]:
images = sorted(p for p in SOURCE_DIR.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
print(f'Imagini gasite: {len(images)}')

all_detections = []
material_counter = Counter()

csv_path = OUTPUT_DIR / 'detections.csv'
with csv_path.open('w', encoding='utf-8', newline='') as fh:
    writer = csv.writer(fh)
    writer.writerow(['image_path','detection_index','x1','y1','x2','y2',
                     'det_score','material_name','material_score'])

    for img_path in images:
        frame = cv2.imread(str(img_path))
        if frame is None: continue

        detections = detect_and_classify(
            frame, detector, classifier,
            DET_CONF, DET_IMGSZ, CLS_IMGSZ, cls_names
        )
        all_detections.append((img_path, frame, detections))
        for d in detections:
            material_counter[d['material_name']] += 1
            l, t, r, b = d['box']
            writer.writerow([img_path.as_posix(), d['index'], l, t, r, b,
                             d['det_score'], d['material_name'], d['material_score']])

total_det = sum(len(d[2]) for d in all_detections)
print(f'Imagini procesate  : {len(all_detections)}')
print(f'Total detectii     : {total_det}')
print(f'CSV salvat         : {csv_path}')
print(f'\nDistributie materiale:')
for mat, cnt in material_counter.most_common():
    print(f'  {mat:<10}: {cnt}')

---
## Distributie Materiale Detectate

In [ ]:
if material_counter:
    labels = list(material_counter.keys())
    values = list(material_counter.values())
    colors = ['#4e9af1','#f17c4e','#62c370','#f1c94e','#a16af1']

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    ax1.bar(labels, values, color=colors[:len(labels)], edgecolor='white', linewidth=0.7)
    ax1.set_title('Numar detectii per material')
    ax1.set_ylabel('Detectii')
    ax1.set_xlabel('Material')
    ax1.tick_params(axis='x', rotation=15)

    ax2.pie(values, labels=labels, colors=colors[:len(labels)], autopct='%1.1f%%',
            startangle=140, pctdistance=0.8)
    ax2.set_title('Distributie procentuala materiale')

    plt.suptitle(f'Two-Stage -- {len(images)} imagini  |  {total_det} detectii totale', fontsize=12)
    plt.tight_layout()

    chart_path = OUTPUT_DIR / 'material_distribution.png'
    plt.savefig(str(chart_path), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Grafic salvat: {chart_path}')
else:
    print('Nicio detectie -- verifica pragul de confidenta (DET_CONF) sau modelul.')

---
## Vizualizare imagini adnotate (Two-Stage)
Afiseaza bounding box-uri + material + confidence pentru primele N imagini cu cele mai multe detectii.

In [ ]:
N_PREVIEW = 6
MATERIAL_COLORS = {
    'glass':   (78,  154, 241),
    'metal':   (241, 124,  78),
    'other':   ( 98, 195, 112),
    'paper':   (241, 201,  78),
    'plastic': (161, 106, 241),
    'unknown': (200, 200, 200),
}

# Selecteaza automat imagini cu cele mai multe detectii
preview_samples = sorted([s for s in all_detections if s[2]], key=lambda x: -len(x[2]))[:N_PREVIEW]
if not preview_samples:
    preview_samples = all_detections[:N_PREVIEW]

if not preview_samples:
    print('Nicio imagine disponibila pentru preview.')
else:
    cols = 3
    n = len(preview_samples)
    rows_n = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows_n, cols, figsize=(16, 5*rows_n))
    axes = np.array(axes).flatten()

    for ax, (img_path, frame, dets) in zip(axes, preview_samples):
        annotated = draw_detections(frame.copy(), dets, fps=0.0, max_labels=MAX_LABELS, line_width=LINE_WIDTH)
        ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        ax.set_title(f'{img_path.name}  [{len(dets)} det.]', fontsize=9)
        ax.axis('off')

    for ax in axes[n:]:
        ax.axis('off')

    plt.suptitle('Rezultate Two-Stage -- Detector + Clasificator Materiale', fontsize=13)
    plt.tight_layout()
    plt.show()

---
## Salvare imagini adnotate

In [ ]:
annotated_dir = OUTPUT_DIR / 'annotated'
annotated_dir.mkdir(exist_ok=True)

saved = 0
for img_path, frame, dets in all_detections:
    annotated = draw_detections(frame.copy(), dets, fps=0.0, max_labels=MAX_LABELS, line_width=LINE_WIDTH)
    cv2.imwrite(str(annotated_dir / img_path.name), annotated)
    saved += 1

print(f'{saved} imagini adnotate salvate in: {annotated_dir}')

---
## Inferenta pe o singura imagine
Testeaza rapid pe orice imagine de pe disc.

In [ ]:
# Cauta prima imagine disponibila din test set
candidates = sorted(SOURCE_DIR.iterdir()) if SOURCE_DIR.is_dir() else []
first_img = next((p for p in candidates if p.suffix.lower() in IMAGE_EXTS), None)

if first_img is None:
    print('Nicio imagine de test gasita.')
else:
    frame = cv2.imread(str(first_img))
    dets  = detect_and_classify(frame, detector, classifier, DET_CONF, DET_IMGSZ, CLS_IMGSZ, cls_names)
    ann   = draw_detections(frame.copy(), dets, fps=0.0, max_labels=5, line_width=2)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Original')
    axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f'Two-Stage ({len(dets)} detectii)')
    axes[1].axis('off')
    plt.suptitle(first_img.name, fontsize=11)
    plt.tight_layout()
    plt.show()

    print(f'\nDetectii ({len(dets)}):')
    for d in dets:
        print(f'  [{d["index"]}] {d["material_name"]:<10} conf={d["material_score"]:.2f}  box={d["box"]}')